CREACIÓN DE MÁSCARAS

In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from datetime import datetime
import pydicom
from pathlib import Path
import matplotlib.pyplot as plt
import os
import napari
import numpy as np
import cv2
import SimpleITK as sitk
import json

In [2]:
import numpy as np
import cv2
import SimpleITK as sitk

# Cargar CT

def load_dicom_folder(path):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(path)
    reader.SetFileNames(dicom_names)
    return reader.Execute()


def sitk_to_numpy(image): # convierte el volumen 3D a numpy (x,y,z)
    return sitk.GetArrayFromImage(image)


# Crear máscara para UNA slice

def crear_mascara_fisica(slice_idx, shape, esferas_mm, spacing):
    h, w = shape
    sx, sy, sz = spacing
    mask = np.zeros((h, w), dtype=np.uint8)

    for s in esferas_mm:
        cz_mm, cy_mm, cx_mm = s["center_mm"]
        R_mm = s["radius_mm"]

        cz = cz_mm / sz
        cy = cy_mm / sy
        cx = cx_mm / sx
        R = R_mm / sx

        dz = abs(slice_idx - cz)
        if dz > R:
            continue

        r_slice = int(np.sqrt(R**2 - dz**2))

        cv2.circle(mask, (int(cx), int(cy)), r_slice, 255, -1)

    return mask


# MAIN

if __name__ == "__main__":

    ct_path = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\CT"

    ct_sitk = load_dicom_folder(ct_path)
    ct_np = sitk_to_numpy(ct_sitk)

    spacing = ct_sitk.GetSpacing()  # (x, y, z)
    nz, ny, nx = ct_np.shape

    # GEOMETRÍA REAL DEL PHANTOM

    diametros_mm = [10, 13, 17, 22, 28, 37]
    radio_anillo_mm = 114 / 2  # ≈ 57 mm

    centro_x_mm = (nx * spacing[0]) / 2
    centro_y_mm = (ny * spacing[1]) / 2
    centro_z_mm = (nz // 2) * spacing[2] + 70  # 70 mm desde montaje

    angulos = np.linspace(0, 2*np.pi, len(diametros_mm), endpoint=False)

    esferas_mm = []
    for d, ang in zip(diametros_mm, angulos):
        cx = centro_x_mm + radio_anillo_mm * np.cos(ang)
        cy = centro_y_mm + radio_anillo_mm * np.sin(ang)

        esferas_mm.append({
            "center_mm": (centro_z_mm, cy, cx),
            "radius_mm": d / 2
        })

    # VISOR SOLO MÁSCARA

    window = "Mascara Phantom"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)

    slice_idx = nz // 2

    def update(val):
        idx = cv2.getTrackbarPos("Slice", window)
        mask = crear_mascara_fisica(idx, (ny, nx), esferas_mm, spacing)
        cv2.imshow(window, mask)

    cv2.createTrackbar("Slice", window, slice_idx, nz - 1, update)
    update(slice_idx)

    cv2.waitKey(0)
    cv2.destroyAllWindows()
